In [23]:
from google import genai

client = genai.Client()

def llm(prompt):
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )
    return response.text

In [3]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

question = "I just discovered the course. Can I join now?"

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
answer = llm(prompt)
print(answer)

Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [4]:
client.close()

In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [19]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [29]:
question = "I just discovered the course. Can I join now?"
search_results = search(question)

prompt = build_prompt(question, search_results)

response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Yes, you can still join. You can start learning and submitting homework whenever you want, provided the submission forms are still open. 

However, please note that if you wish to receive a certificate, you must submit your project while the course is still accepting submissions and participate in the live cohort to complete the required peer reviews. Certificates are not awarded for self-paced study.""",
        thought_signature=b'\x124\n2\x01\x0c9\xd6\xc7\x14\xea;\xee\x872\xe10P\x81\xde\xcdw\x08A\xcfAl\x80\x1d\xad\xf5\xa0\xf4j\xa7y\x1e1\xd3\xc5\xf9\xcak\xbc\xcf\x891\x06V\xdc\xf5$1:'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='XqY5atL2BrWI1MkPv6fdyAc' usage_metadata=GenerateContentResponseUsageMetadata(
  ca

In [36]:
response.usage_metadata.prompt_token_count

514

In [37]:
input_price = 0.25 / 1_000_000
output_price = 1.50 / 1_000_000

cost = (
    response.usage_metadata.prompt_token_count * input_price +
    response.usage_metadata.candidates_token_count * output_price
)

cost

0.0002425